# LangGraph architecture (live)

Diagrams are rendered from **compiled** LangGraph objects — not hand-drawn.

- **Product code:** `src/explain_my_option/` graph + pipeline modules
- **Package entry:** [getting_started.ipynb](getting_started.ipynb)

**How to run**

1. Kernel = repo `.venv`.
2. **Run → Run All Cells**.
3. PNG needs network (mermaid.ink). Mermaid text is always printed as fallback.

Book-first parent with `Send` fan-out → leg subgraph (residual loop + verifier as graph edges). Single-leg CLI/UI is one leg in the same graph.


In [ ]:
from pathlib import Path
import sys

from IPython.display import Image, Markdown, display

here = Path.cwd().resolve()
tests_dir = None
for candidate in [here, *here.parents]:
    if (candidate / "tests" / "bootstrap.py").is_file() and (
        candidate / "src" / "explain_my_option"
    ).is_dir():
        tests_dir = candidate / "tests"
        break
if tests_dir is None:
    raise RuntimeError("Open this notebook from the ExplainMyOption repo.")
if str(tests_dir) not in sys.path:
    sys.path.insert(0, str(tests_dir))

from bootstrap import find_repo_root, install

ROOT = install()
assert ROOT == find_repo_root()
print("repo root:", ROOT)


def show_graph(compiled, title: str, *, xray: bool = False) -> None:
    suffix = " (xray)" if xray else ""
    display(Markdown(f"## {title}{suffix}"))
    graph = compiled.get_graph(xray=xray)
    try:
        display(Image(graph.draw_mermaid_png()))
    except Exception as exc:
        display(Markdown(
            f"PNG failed (often mermaid.ink network): `{type(exc).__name__}: {exc}`"
        ))
        display(Markdown("Mermaid source:"))
        print(graph.draw_mermaid())


## Agent state schemas

Three `TypedDict` layers (no `messages` dump):

| State | Module | Role |
|-------|--------|------|
| `PortfolioState` | `src/explain_my_option/pipeline/portfolio_graph.py` | Book parent: `book`, `leg_results` (reducer), `book_report` |
| `LegState` | `src/explain_my_option/pipeline/leg_graph.py` | One leg: extends `OptionState` + residual/verifier control fields |
| `OptionState` | `src/explain_my_option/graph/state.py` | Single-leg compatibility view for `app.py` / `run_pipeline` |

`run_pipeline()` still returns `OptionState`; internally it runs the unified book graph with one leg.


## 1. Product entry (`build_graph` / `run_pipeline`)

Parent graph: `require_openai` → `Send` fan-out → `leg_branch` → `aggregate_book`.

Use `xray=True` to expand nested nodes (fetch → quant → blotter → diagnostic_pass → residual loop → search → digest → challenge → synthesize → verify).


In [ ]:
from explain_my_option.agent_graph import build_graph

product = build_graph()
show_graph(product, "Unified product graph (book parent)")
show_graph(product, "Unified product graph (book parent)", xray=True)


## 2. Leg diagnosis subgraph (`build_leg_diagnosis_subgraph`)

Explicit graph edges for the residual loop (`residual_gate` → `react_plan` → `react_tool_exec` → `budget_gate`) and verifier (`verify` → `revise_synthesis` or `terminal_unexplained_break`).


In [ ]:
from explain_my_option.pipeline.book_schema import BookLegSpec
from explain_my_option.pipeline.config import PipelineConfig
from explain_my_option.pipeline.leg_graph import build_leg_diagnosis_subgraph

leg = BookLegSpec(leg_id="demo", fixture="vol_crush", ticker="AAPL")
leg_graph = build_leg_diagnosis_subgraph(
    config=PipelineConfig(require_openai=False),
)
show_graph(leg_graph, "Leg diagnosis subgraph")
show_graph(leg_graph, "Leg diagnosis subgraph", xray=True)


## 3. Book parent (`build_book_graph`)

Dashed `require_openai -.-> leg_branch` is LangGraph `Send` fan-out (one `leg_branch` invoke per `BookLegSpec`).


In [ ]:
from explain_my_option.pipeline.portfolio_graph import build_book_graph

book_graph = build_book_graph()
show_graph(book_graph, "Book parent (Send fan-out)")
show_graph(book_graph, "Book parent (Send fan-out)", xray=True)


## Optional: Mermaid source (offline / docs)

Paste into GitHub, Notion, or the Streamlit topology tab in `app.py`. Helpers: `src/explain_my_option/graph/topology.py`.


In [ ]:
from explain_my_option.graph.topology import book_mermaid, leg_mermaid, pipeline_mermaid

print("=== pipeline (product) ===")
print(pipeline_mermaid(xray=True))
print("=== leg subgraph ===")
print(leg_mermaid(xray=True))
print("=== book parent ===")
print(book_mermaid())
